# 1.2.2 Pandas

DataFrame operations — load, clean, feature engineer, groupby, merge, query — on Titanic dataset.

In [ ]:
import pandas as pd, numpy as np, urllib.request
from pathlib import Path
DATA=Path('data'); DATA.mkdir(exist_ok=True)
dest=DATA/'titanic.csv'
if not dest.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv', dest)
df=pd.read_csv(dest)
print(df.shape); df.head()

In [ ]:
# Missing data
print(df.isnull().sum()[df.isnull().sum()>0])
df['Age']=df['Age'].fillna(df['Age'].median())
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
print('Nulls after fill:', df.isnull().sum().sum())

In [ ]:
# Feature engineering (method chaining)
df=df.assign(
    Title=df['Name'].str.extract(r',\s*([^\.]+)\.').iloc[:,0].str.strip(),
    FamilySize=df['SibSp']+df['Parch']+1,
    IsAlone=(df['SibSp']+df['Parch']==0).astype(int),
    LogFare=np.log1p(df['Fare']),
    AgeBin=pd.cut(df['Age'],bins=[0,12,18,35,60,100],labels=['child','teen','adult','middle','senior'])
)
df[['Title','FamilySize','IsAlone','LogFare','AgeBin']].head()

In [ ]:
# GroupBy
df.groupby(['Pclass','Sex'])['Survived'].agg(['count','mean']).round(3)

In [ ]:
# Pivot table
df.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean').round(3)

In [ ]:
# Visualization
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
df.groupby('Pclass')['Survived'].mean().plot(kind='bar',ax=axes[0],title='Survival by Class')
df['Fare'].clip(upper=200).plot(kind='hist',bins=40,ax=axes[1],title='Fare Distribution')
plt.tight_layout(); plt.show()